# EMA 7/21 Cross — Directional Options Backtest

Evaluates buying **calls** on bullish EMA crossovers and **puts** on bearish crossovers against real Deribit chain snapshots.

| Signal | Trade |
|--------|-------|
| EMA-7 crosses **above** EMA-21 | Buy **call** nearest to target delta / DTE |
| EMA-7 crosses **below** EMA-21 | Buy **put** nearest to target delta / DTE |

Exit is mark-to-market (bid) after `HOLD_DAYS`. If hold extends past expiry, intrinsic value is used instead.

**Workflow:**
1. Run **cell 2** once to set parameters
2. Run **cells 3–4** once to load data and build features
3. Run **cell 5** to define helpers
4. Run **cell 6** to evaluate and print results
5. Use the **playground cell** at the bottom to sweep parameters freely

In [1]:
# ── PARAMETERS — edit these and re-run the evaluation cell ──────────────────
DTE        = 3      # days-to-expiry at entry          (try: 1, 2, 3, 4, 7)
DELTA      = 0.30   # target absolute delta per leg     (try: 0.20, 0.25, 0.30, 0.40)
HOLD_DAYS  = 3      # hold duration in calendar days    (try: 1, 2, 3, 5, 7)
#                     Note: if HOLD_DAYS >= DTE the position is held to expiry
#                     and intrinsic value is used at exit.

In [2]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from cryocore.instruments import Symbol
from cryoquant import config
from cryoquant.data.loader import load
from cryoquant.features.builders import DatasetRef, DailyEmaCrossFeatures
from cryoquant.backtest.option_lookup import (
    OptionResult,
    _best_leg,
    _load_chain_df,
    _spot_price_on_date,
)
from cryoquant.backtest.reports import render_option_result

CHAINS_DIR = config.CRYOBACKTESTER_DATA_DIR
REPORTS    = Path("../reports")
REPORTS.mkdir(exist_ok=True)

# Chain data covers 2025-04-11 → 2026-05-12; load a month earlier for EMA warmup
START = datetime(2025, 3, 1, tzinfo=timezone.utc)
END   = datetime.now(timezone.utc)

print(f"Chains dir : {CHAINS_DIR}")
print(f"Bar range  : {START.date()} → {END.date()}")

Chains dir : /Users/ulrikdeichsel/CryoBacktester/backtester/data
Bar range  : 2025-03-01 → 2026-05-19


In [3]:
sym    = Symbol("binance.spot", "BTCUSDT")
df_raw = load(sym, "1d", START, END)

ref      = DatasetRef(sym, "1d")
X        = DailyEmaCrossFeatures().build({ref: df_raw})
new_cols = [c for c in X.columns if c not in df_raw.columns]
bars     = df_raw.join(X[new_cols])

cross_up_times   = bars.index[X["cross_up"].fillna(False).astype(bool)]
cross_down_times = bars.index[X["cross_down"].fillna(False).astype(bool)]

print(f"✓ {len(bars)} daily bars  ({bars.index[0].date()} → {bars.index[-1].date()})")
print(f"  Up crosses  : {len(cross_up_times)}  →  {list(cross_up_times.date)}")
print(f"  Down crosses: {len(cross_down_times)}  →  {list(cross_down_times.date)}")

✓ 444 daily bars  (2025-03-01 → 2026-05-18)
  Up crosses  : 12  →  [datetime.date(2025, 3, 25), datetime.date(2025, 4, 15), datetime.date(2025, 6, 9), datetime.date(2025, 6, 27), datetime.date(2025, 8, 10), datetime.date(2025, 9, 12), datetime.date(2025, 10, 2), datetime.date(2026, 1, 4), datetime.date(2026, 3, 6), datetime.date(2026, 3, 12), datetime.date(2026, 3, 25), datetime.date(2026, 4, 8)]
  Down crosses: 11  →  [datetime.date(2025, 3, 30), datetime.date(2025, 6, 6), datetime.date(2025, 6, 19), datetime.date(2025, 8, 3), datetime.date(2025, 8, 20), datetime.date(2025, 9, 25), datetime.date(2025, 10, 13), datetime.date(2026, 1, 22), datetime.date(2026, 3, 7), datetime.date(2026, 3, 23), datetime.date(2026, 3, 27)]


## Helper functions

In [4]:
def eval_leg(fire_timestamps, *, is_call, dte, delta, hold_days, chains_dir=CHAINS_DIR):
    """Evaluate single-leg directional trades for all fire timestamps.

    Returns (pnl_pct, entry_costs_usd, dte_actual).
    P&L is a fraction of entry cost — positive = profit.
    """
    pnl_pct, entry_costs_usd, dte_actual = [], [], []

    for ts in fire_timestamps:
        fire_date   = ts.date()
        expiry_date = fire_date + timedelta(days=dte)

        entry_chain = _load_chain_df(chains_dir, fire_date)
        if entry_chain is None:
            continue

        spot = _spot_price_on_date(chains_dir, fire_date)
        if spot is None:
            spot = float(bars.loc[ts, "close"]) if ts in bars.index else None
        if spot is None:
            continue

        leg = _best_leg(entry_chain, expiry_date, delta, is_call=is_call)
        if leg is None:
            continue

        entry_ask = float(leg["ask_price"])
        entry_cost_usd = entry_ask * spot
        if entry_cost_usd <= 0:
            continue

        strike    = float(leg["strike"])
        exit_date = fire_date + timedelta(days=hold_days)
        at_expiry = exit_date >= expiry_date
        if at_expiry:
            exit_date = expiry_date

        if not at_expiry:
            exit_chain = _load_chain_df(chains_dir, exit_date)
            exit_spot  = _spot_price_on_date(chains_dir, exit_date)
            if exit_chain is None or exit_spot is None:
                continue
            exit_leg = _best_leg(exit_chain, expiry_date, delta, is_call=is_call)
            if exit_leg is not None:
                exit_bid = float(exit_leg["bid_price"])
                if exit_bid <= 0:
                    exit_bid = float(exit_leg["ask_price"]) * 0.9
                exit_value = exit_bid
            else:
                exit_value = (max(0.0, exit_spot - strike) / exit_spot if is_call
                              else max(0.0, strike - exit_spot) / exit_spot)
        else:
            exp_spot = _spot_price_on_date(chains_dir, expiry_date) or spot
            exit_value = (max(0.0, exp_spot - strike) / exp_spot if is_call
                          else max(0.0, strike - exp_spot) / exp_spot)

        pnl_pct.append((exit_value - entry_ask) / entry_ask)
        entry_costs_usd.append(entry_cost_usd)
        dte_actual.append((expiry_date - fire_date).days)

    return pnl_pct, entry_costs_usd, dte_actual


def make_result(n_fires, pnl_pct, entry_costs_usd, dte_actual):
    n = len(pnl_pct)
    win_rate   = float(np.mean(np.array(pnl_pct) > 0)) if n > 0 else float("nan")
    expectancy = float(np.mean(pnl_pct))                if n > 0 else float("nan")
    return OptionResult(
        fires_evaluated=n_fires, fires_with_data=n,
        pnl_pct=pnl_pct, win_rate=win_rate, expectancy=expectancy,
        entry_costs_usd=entry_costs_usd, dte_actual=dte_actual,
    )


def print_result(label, result):
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Fires evaluated : {result.fires_evaluated}")
    print(f"  Fires with data : {result.fires_with_data}")
    if result.fires_with_data > 0:
        arr = np.array(result.pnl_pct)
        print(f"  Win rate        : {result.win_rate:.1%}")
        print(f"  Expectancy      : {result.expectancy:+.2%}")
        print(f"  Median P&L      : {float(np.median(arr)):+.2%}")
        print(f"  Best / Worst    : {arr.max():+.2%} / {arr.min():+.2%}")
        print(f"  Median cost     : ${float(np.median(result.entry_costs_usd)):,.0f}")
    else:
        print("  No chain data found for these fire dates.")

print("✓ helpers defined")

✓ helpers defined


## Evaluate — default parameters

Uses `DTE`, `DELTA`, and `HOLD_DAYS` from the parameters cell. Re-run this cell after changing them.

In [5]:
call_pnl, call_costs, call_dte = eval_leg(
    cross_up_times, is_call=True, dte=DTE, delta=DELTA, hold_days=HOLD_DAYS
)
put_pnl, put_costs, put_dte = eval_leg(
    cross_down_times, is_call=False, dte=DTE, delta=DELTA, hold_days=HOLD_DAYS
)

result_calls = make_result(len(cross_up_times),   call_pnl, call_costs, call_dte)
result_puts  = make_result(len(cross_down_times),  put_pnl,  put_costs,  put_dte)

print_result(f"CALLS — EMA cross up   (DTE={DTE}  delta={DELTA}  hold={HOLD_DAYS}d)", result_calls)
print_result(f"PUTS  — EMA cross down (DTE={DTE}  delta={DELTA}  hold={HOLD_DAYS}d)", result_puts)

# Write HTML reports
tag = f"dte{DTE}_d{int(DELTA*100)}_h{HOLD_DAYS*24}"
render_option_result(result_calls, str(REPORTS / f"ema_cross_calls_{tag}.html"), dte=DTE, delta=DELTA)
render_option_result(result_puts,  str(REPORTS / f"ema_cross_puts_{tag}.html"),  dte=DTE, delta=DELTA)
print(f"\n✓ Reports written to reports/ema_cross_calls_{tag}.html  +  puts")


────────────────────────────────────────────────────────────
  CALLS — EMA cross up   (DTE=3  delta=0.3  hold=3d)
────────────────────────────────────────────────────────────
  Fires evaluated : 12
  Fires with data : 11
  Win rate        : 27.3%
  Expectancy      : -0.50%
  Median P&L      : -100.00%
  Best / Worst    : +395.88% / -100.00%
  Median cost     : $641

────────────────────────────────────────────────────────────
  PUTS  — EMA cross down (DTE=3  delta=0.3  hold=3d)
────────────────────────────────────────────────────────────
  Fires evaluated : 11
  Fires with data : 10
  Win rate        : 30.0%
  Expectancy      : +47.63%
  Median P&L      : -85.48%
  Best / Worst    : +723.65% / -100.00%
  Median cost     : $668

✓ Reports written to reports/ema_cross_calls_dte3_d30_h72.html  +  puts


## Let's play with it

Sweep any combination of DTE, delta, and hold period. Each row in the results table is independent — no need to re-run earlier cells.

In [6]:
combos = [
    # (dte, delta, hold_days)
    (1, 0.30, 1),
    (2, 0.30, 2),
    (3, 0.30, 3),   # ← default
    (3, 0.25, 3),
    (3, 0.40, 3),
    (7, 0.30, 5),
    (7, 0.25, 7),   # hold to expiry
]

rows = []
for dte, delta, hold in combos:
    cp, cc, cd = eval_leg(cross_up_times,   is_call=True,  dte=dte, delta=delta, hold_days=hold)
    pp, pc, pd_ = eval_leg(cross_down_times, is_call=False, dte=dte, delta=delta, hold_days=hold)

    def _stats(pnl, costs, n_fires):
        n = len(pnl)
        if n == 0:
            return {"fires": n_fires, "data": 0, "win%": "-", "exp": "-", "med_cost": "-"}
        arr = np.array(pnl)
        return {
            "fires": n_fires, "data": n,
            "win%":  f"{np.mean(arr > 0):.0%}",
            "exp":   f"{np.mean(arr):+.1%}",
            "med_cost": f"${np.median(costs):,.0f}",
        }

    cs = _stats(cp, cc, len(cross_up_times))
    ps = _stats(pp, pc, len(cross_down_times))
    rows.append({
        "dte": dte, "delta": delta, "hold": hold,
        "call_win%": cs["win%"], "call_exp": cs["exp"], "call_med_cost": cs["med_cost"],
        "put_win%":  ps["win%"], "put_exp":  ps["exp"], "put_med_cost":  ps["med_cost"],
    })

df_results = pd.DataFrame(rows)
df_results

,dte,delta,hold,call_win%,call_exp,call_med_cost,put_win%,put_exp,put_med_cost
0,1,0.30,1,27%,+82.4%,$298,60%,+25.0%,$399
1,2,0.30,2,18%,-69.4%,$511,40%,-5.3%,$543
2,3,0.30,3,27%,-0.5%,$641,30%,+47.6%,$668
3,3,0.25,3,27%,+7.6%,$483,30%,-0.0%,$555
4,3,0.40,3,27%,+0.7%,$853,50%,+43.3%,$942
5,7,0.30,5,0%,-44.5%,$920,0%,-54.0%,"$1,148"
6,7,0.25,7,0%,-100.0%,$750,0%,-100.0%,$823
